# Functional Analysis Dataset Creation

**Instructor-only notebook** -- generates `functional_analysis_data.csv` for the Week 8 lab.

## Design Rationale

This notebook creates a simulated multielement functional analysis dataset with:

- **4 conditions** cycled in a fixed rotation: attention, escape, tangible, play (control).
- **10 complete cycles** (40 total sessions) to give students enough data for Bayesian updating.
- **Clearly attention-maintained** problem behavior: the attention condition produces rates roughly 8--11 responses/min, whereas escape (~0.6--1.5), tangible (~1.6--2.5), and play (~0.1--0.6) are all low.
- Rates include realistic session-to-session variability within each condition.

The data are structured so that even after a few cycles, a Bayesian update from a uniform prior will shift strongly toward "attention" as the most probable function.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(123)

## Condition Parameters

Response rates for each condition are drawn from the following distributions:

| Condition  | Mean rate | SD   | Notes                          |
|------------|-----------|------|--------------------------------|
| Attention  | 9.3       | 1.0  | Elevated -- this is the function |
| Escape     | 1.05      | 0.3  | Low, operant level             |
| Tangible   | 2.0       | 0.3  | Slightly above control         |
| Play       | 0.3       | 0.15 | Control condition, near-zero   |

The tangible condition is set slightly above play/escape to give students something to think about, but the separation between attention and all other conditions is unambiguous.

In [ ]:
# Exact values matching the target CSV
# These were hand-specified to produce a clean, interpretable FA graph
# while maintaining realistic session-to-session variability.

condition_order = ["attention", "escape", "tangible", "play"]
n_cycles = 10

rates = {
    "attention": [8.4, 9.1, 7.9, 10.2, 8.7, 9.5, 11.0, 8.2, 9.8, 10.5],
    "escape":    [1.2, 0.8, 1.5, 0.6, 1.1, 0.9, 1.3, 0.7, 1.0, 1.4],
    "tangible":  [2.1, 1.7, 2.4, 1.9, 2.0, 1.6, 2.3, 1.8, 2.5, 2.2],
    "play":      [0.3, 0.5, 0.2, 0.4, 0.1, 0.3, 0.6, 0.2, 0.4, 0.3],
}

print("Condition means:")
for cond in condition_order:
    print(f"  {cond:10s}: M = {np.mean(rates[cond]):.2f}, SD = {np.std(rates[cond]):.2f}")

## Verify the Attention-Maintained Pattern

Every attention session rate should exceed the maximum rate in any other condition in the same cycle.

In [ ]:
for i in range(n_cycles):
    attn = rates["attention"][i]
    others_max = max(rates["escape"][i], rates["tangible"][i], rates["play"][i])
    assert attn > others_max, f"Cycle {i}: attention ({attn}) not highest!"

print("All cycles confirmed: attention rate is always the highest condition.")

## Build and Save the CSV

Sessions are numbered 1--40. Conditions rotate in the fixed order: attention, escape, tangible, play.

In [ ]:
rows = []
session = 1
for cycle in range(n_cycles):
    for cond in condition_order:
        rows.append({
            "session": session,
            "condition": cond,
            "rate_per_min": rates[cond][cycle],
        })
        session += 1

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nShape: {df.shape}")

In [ ]:
df.to_csv("functional_analysis_data.csv", index=False)
print("Saved functional_analysis_data.csv")